# 03 Scaling Runtime

This notebook visualizes runtime and memory scaling with:

- `M`: number of transmit antennas
- `K`: number of users

Input: `result/table/03_scaling_runtime.csv`

Outputs: PDF figures in `result/figure/` and aggregated tables in `result/table/`.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path(__file__).resolve().parents[1]
tbl_dir = ROOT / "result" / "table"
fig_dir = ROOT / "result" / "figure"
fig_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(tbl_dir / "03_scaling_runtime.csv")
df.head()


In [ ]:
# Aggregate mean/std over runs per (M,K)
metrics = ["avg_slot_ms", "peak_mem_mb", "avg_sum_queue", "avg_sense_u"]

agg = df.groupby(["M", "K"])[metrics].agg(["mean", "std"]).reset_index()
agg.columns = ["M", "K"] + [f"{m}_{s}" for m in metrics for s in ["mean", "std"]]

out_table = tbl_dir / "03_scaling_runtime_table.csv"
agg.to_csv(out_table, index=False)
print("Saved table:", out_table)

agg.head()


In [ ]:
# Heatmap of runtime
piv = agg.pivot(index="K", columns="M", values="avg_slot_ms_mean")
plt.figure()
im = plt.imshow(piv.values, aspect="auto", origin="lower")
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.xticks(range(len(piv.columns)), piv.columns)
plt.yticks(range(len(piv.index)), piv.index)
plt.xlabel("M (Tx antennas)")
plt.ylabel("K (users)")
plt.title("Average per-slot runtime (ms, mean)")
out = fig_dir / "03_heatmap_runtime.pdf"
plt.savefig(out, format="pdf", bbox_inches="tight")
print("Saved:", out)


In [ ]:
# Runtime vs M for each K
plt.figure()
for k in sorted(agg["K"].unique()):
    sub = agg[agg["K"] == k].sort_values("M")
    plt.plot(sub["M"], sub["avg_slot_ms_mean"], marker="o", label=f"K={k}")
plt.xlabel("M (Tx antennas)")
plt.ylabel("Average per-slot runtime (ms)")
plt.legend()
plt.grid(True, alpha=0.3)
out = fig_dir / "03_runtime_vs_M.pdf"
plt.savefig(out, format="pdf", bbox_inches="tight")
print("Saved:", out)


In [ ]:
# Peak GPU memory vs M for each K (if available)
if np.isfinite(agg["peak_mem_mb_mean"]).any():
    plt.figure()
    for k in sorted(agg["K"].unique()):
        sub = agg[agg["K"] == k].sort_values("M")
        plt.plot(sub["M"], sub["peak_mem_mb_mean"], marker="o", label=f"K={k}")
    plt.xlabel("M (Tx antennas)")
    plt.ylabel("Peak GPU memory (MB)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    out = fig_dir / "03_peak_memory_vs_M.pdf"
    plt.savefig(out, format="pdf", bbox_inches="tight")
    print("Saved:", out)
else:
    print("No GPU memory stats found (likely ran on CPU).")


In [ ]:
# Bubble-style scatter: runtime vs performance with bubble size proportional to M*K
size = (agg["M"] * agg["K"]).values.astype(float)
size = 50.0 * (size / np.max(size))

plt.figure()
plt.scatter(agg["avg_slot_ms_mean"], agg["avg_sum_queue_mean"], s=size)
for _, r in agg.iterrows():
    plt.annotate(f"M{int(r['M'])},K{int(r['K'])}", (r["avg_slot_ms_mean"], r["avg_sum_queue_mean"]))
plt.xlabel("Avg per-slot runtime (ms)")
plt.ylabel("Avg total queue")
plt.grid(True, alpha=0.3)
out = fig_dir / "03_bubble_runtime_vs_queue.pdf"
plt.savefig(out, format="pdf", bbox_inches="tight")
print("Saved:", out)
